In [9]:
import pandas as pd
import numpy as np
import os
 
pd.set_option("display.max_columns", 60)
 
RAW_DIR = "../data/raw_data"
PROC_DIR = "../data/processed"
 
employees_df = pd.read_csv(f"{RAW_DIR}/employees_raw.csv")
region_df = pd.read_csv(f"{RAW_DIR}/region_benefit_profiles.csv")
 
print("=== EMPLOYEES PROFILE ===")
print(employees_df.shape)
print(employees_df.dtypes)
print("\nMissingness:\n", employees_df.isna().sum())
print("\nCardinality:\n", employees_df.nunique())
print("\nDuplicate rows (full):", employees_df.duplicated().sum())
print("\nDuplicate employee_id:", employees_df["employee_id"].duplicated().sum())
 
# conflicting labels
dup_labels = employees_df.groupby("employee_id")["enrolled"].nunique()
conflict_ids = dup_labels[dup_labels > 1].index
print("\nConflicting-label employee_ids:", list(conflict_ids))
 
print("\n=== REGION PROFILE ===")
print(region_df.shape)
print(region_df.dtypes)
print(region_df.isna().sum())
print(region_df["region"].nunique(), region_df["region"].unique())

=== EMPLOYEES PROFILE ===
(10008, 18)
employee_id                  int64
age                          int64
gender                      object
marital_status              object
salary                     float64
employment_type             object
region                      object
has_dependents              object
tenure_years               float64
enrolled                     int64
application_date            object
last_contact_date           object
last_contact_channel        object
plan_tier_requested         object
broker_channel              object
prior_year_enrolled          int64
legacy_propensity_score    float64
outreach_notes              object
dtype: object

Missingness:
 employee_id                   0
age                           0
gender                        0
marital_status                0
salary                        0
employment_type               0
region                        0
has_dependents                0
tenure_years                  0
enrolled       

Data Cleaning

In [18]:
# ---------------- Deduplication policy ----------------
before = len(employees_df)
conflicting_ids = employees_df.groupby("employee_id")["enrolled"].nunique()
conflicting_ids = conflicting_ids[conflicting_ids > 1].index
n_conflict_rows = employees_df["employee_id"].isin(conflicting_ids).sum()
 
employees_df = employees_df[~employees_df["employee_id"].isin(conflicting_ids)].copy()
n_exact_dupe = employees_df.duplicated(subset=["employee_id"], keep="first").sum()
employees_df = employees_df.drop_duplicates(subset=["employee_id"], keep="first")
after = len(employees_df)
print(f"\nDedup: dropped {n_conflict_rows} conflicting-label rows, "
      f"{n_exact_dupe} exact repeat rows. {before} -> {after}")



Dedup: dropped 16 conflicting-label rows, 0 exact repeat rows. 10008 -> 9992


In [19]:
# ---------------- Date parsing ----------------
employees_df["application_date"] = pd.to_datetime(
    employees_df["application_date"], format="mixed", errors="coerce"
)
employees_df["last_contact_date"] = pd.to_datetime(
    employees_df["last_contact_date"], format="mixed", errors="coerce"
)
 
employees_df["has_missing_app_date"] = employees_df["application_date"].isna().astype(int)
employees_df["is_contact_after_app"] = (
    employees_df["last_contact_date"] > employees_df["application_date"]
).astype(int)
 
print("\nis_contact_after_app count:", employees_df["is_contact_after_app"].sum())
print("has_missing_app_date count:", employees_df["has_missing_app_date"].sum())


is_contact_after_app count: 337
has_missing_app_date count: 719


In [20]:
# ---------------- Categorical standardization ----------------
channel_map = {
    "email": "Email", "e-mail": "Email", "mail": "Email",
    "call": "Phone", "phone": "Phone",
    "sms": "SMS", "text": "SMS",
    "none": "None", "n/a": "None", "na": "None", "nan": "None", "": "None",
}
employees_df["last_contact_channel"] = (
    employees_df["last_contact_channel"].fillna("None").astype(str).str.strip().str.lower()
    .map(channel_map).fillna("None")
)
 
tier_map = {
    "basic": "Basic", "bronze": "Basic",
    "standard": "Standard", "silver": "Standard", "silver plan": "Standard",
    "premium": "Premium", "premium plan": "Premium", "gold": "Premium", "gold plan": "Premium",
    "unspecified": "Unspecified", "nan": "Unspecified", "": "Unspecified",
}
employees_df["plan_tier_requested"] = (
    employees_df["plan_tier_requested"].fillna("unspecified").astype(str).str.strip().str.lower()
    .map(tier_map).fillna("Unspecified")
)
print("\nChannel values:", employees_df["last_contact_channel"].value_counts().to_dict())
print("Tier values:", employees_df["plan_tier_requested"].value_counts().to_dict())
 


Channel values: {'Email': 4499, 'Phone': 2428, 'None': 1587, 'SMS': 1478}
Tier values: {'Standard': 6484, 'Basic': 2042, 'Premium': 942, 'Unspecified': 524}


In [21]:
# ---------------- Sentinel handling ----------------
employees_df["is_new_hire"] = (employees_df["prior_year_enrolled"] == 1).astype(int)
employees_df["prior_year_enrolled_clean"] = np.where(
    employees_df["prior_year_enrolled"] == 1, 0, employees_df["prior_year_enrolled"]
)
print("\nis_new_hire counts:", employees_df["is_new_hire"].value_counts().to_dict())
 


is_new_hire counts: {0: 6421, 1: 3571}


In [22]:
# ---------------- Impossible value checks ----------------
print("\nAge out of plausible working range (<16 or >75):",
      ((employees_df["age"] < 16) | (employees_df["age"] > 75)).sum())
print("Tenure > age - 16:", (employees_df["tenure_years"] > employees_df["age"] - 16).sum())
 
salary_thresholds = {"Full-time": 15000, "Contract": 18000, "Part-time": 18000}
employees_df["min_salary_allowed"] = employees_df["employment_type"].map(salary_thresholds)
employees_df["is_implausible_salary"] = (
    employees_df["salary"].isna() | (employees_df["salary"] < employees_df["min_salary_allowed"])
).astype(int)
 
valid_salary_mask = employees_df["salary"] >= employees_df["min_salary_allowed"]
median_by_type = employees_df.loc[valid_salary_mask].groupby("employment_type")["salary"].median()
global_median = employees_df.loc[valid_salary_mask, "salary"].median()
 
employees_df["salary_clean"] = employees_df["salary"]
bad_mask = employees_df["is_implausible_salary"] == 1
employees_df.loc[bad_mask, "salary_clean"] = (
    employees_df.loc[bad_mask, "employment_type"].map(median_by_type).fillna(global_median)
)
print("\nImplausible salary rows:", employees_df["is_implausible_salary"].sum())
 
# tenure cap by working-age bound (min working age 18), then winsorize
max_possible_tenure = (employees_df["age"] - 18).clip(lower=0)
employees_df["tenure_years"] = np.minimum(employees_df["tenure_years"], max_possible_tenure)
employees_df["tenure_years"] = employees_df["tenure_years"].round(1)
 
lo, hi = employees_df["salary_clean"].quantile([0.01, 0.99])
employees_df["salary_clean"] = employees_df["salary_clean"].clip(lo, hi)
 


Age out of plausible working range (<16 or >75): 0
Tenure > age - 16: 235

Implausible salary rows: 4


In [23]:
# ---------------- legacy_propensity_score: missingness + leakage check ----------------
print("\nlegacy_propensity_score missing:", employees_df["legacy_propensity_score"].isna().sum())
tmp = employees_df.dropna(subset=["legacy_propensity_score"])
try:
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(tmp["enrolled"], tmp["legacy_propensity_score"])
    print(f"Single-feature AUC of legacy_propensity_score vs enrolled: {auc:.3f}")
except Exception as e:
    print("AUC check skipped:", e)
employees_df["legacy_propensity_score_missing"] = employees_df["legacy_propensity_score"].isna().astype(int)
 


legacy_propensity_score missing: 899
Single-feature AUC of legacy_propensity_score vs enrolled: 1.000


In [24]:
# ---------------- outreach_notes: presence flag only, no content used ----------------
employees_df["has_outreach_note"] = employees_df["outreach_notes"].notna().astype(int)
 

In [39]:
# 1. Create a binary indicator flag for missingness
employees_df['has_applied'] = employees_df['application_date'].notna().astype(int)

# 2. Extract Date Components safely (Categoricals filled with 'Missing', Numerics with -1)
employees_df['app_month'] = (
    employees_df['application_date'].dt.strftime('%B').fillna('Missing')
)
employees_df['app_day_of_week'] = (
    employees_df['application_date'].dt.strftime('%A').fillna('Missing')
)

# 3. Date Difference Features (e.g., days between contact and application)
days_diff = (
    employees_df['application_date'] - employees_df['last_contact_date']
).dt.days
employees_df['days_contact_to_app'] = days_diff.fillna(
    -1
)  # -1 indicates no application or no contact

# 4. Binary Comparison Flags (e.g., contact occurred after application)
employees_df['is_contact_after_app'] = (
    employees_df['last_contact_date'] > employees_df['application_date']
).fillna(False).astype(int)

In [43]:
employees_df[employees_df["application_date"].isna()]
# employees_df.isna().sum()[employees_df.isna().sum() > 0]

,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,application_date,last_contact_date,last_contact_channel,plan_tier_requested,broker_channel,prior_year_enrolled,legacy_propensity_score,outreach_notes,has_missing_app_date,is_contact_after_app,is_new_hire,prior_year_enrolled_clean,min_salary_allowed,is_implausible_salary,salary_clean,legacy_propensity_score_missing,has_outreach_note,days_since_application,days_since_last_contact,contact_to_application_days,month_of_application,day_of_week_application,salary_per_tenure,salary_age_ratio,salary_band,has_applied,app_month,app_day_of_week,days_contact_to_app
8,16183,38,Male,Single,70490.72,Full-time,West,No,2.3,1,NaT,2024-05-16,Phone,Standard,Employer-Sponsored,-1,NaN,Follow-up scheduled,1,0,0,-1,15000,0,70490.72,1,1,NaN,228,NaN,NaN,NaN,21360.824242,1855.018947,3,0,Missing,Missing,-1.0
19,17481,28,Female,Single,72116.53,Part-time,South,Yes,5.6,0,NaT,2024-10-21,Email,Standard,Direct,0,0.072,Declined - cost concern,1,0,0,0,18000,0,72116.53,0,1,NaN,70,NaN,NaN,NaN,10926.746970,2575.590357,3,0,Missing,Missing,-1.0
24,11798,42,Female,Single,91841.05,Full-time,South,No,2.2,1,NaT,2024-11-22,Phone,Premium,Direct,1,0.892,Requested more time,1,0,1,0,15000,0,91841.05,0,1,NaN,38,NaN,NaN,NaN,28700.328125,2186.691667,4,0,Missing,Missing,-1.0
47,10949,58,Male,Married,68129.63,Full-time,South,Yes,1.0,1,NaT,2024-07-24,Email,Standard,Employer-Sponsored,0,0.844,Requested more time,1,0,0,0,15000,0,68129.63,0,1,NaN,159,NaN,NaN,NaN,34064.815000,1174.648793,2,0,Missing,Missing,-1.0
66,13295,62,Female,Single,42866.80,Part-time,West,No,1.6,0,NaT,2024-03-30,Phone,Unspecified,Third-Party,0,0.115,Attended benefits webinar,1,0,0,0,18000,0,42866.80,0,1,NaN,275,NaN,NaN,NaN,16487.230769,691.400000,0,0,Missing,Missing,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9934,14733,38,Female,Single,84107.08,Full-time,South,Yes,3.2,1,NaT,2024-11-03,Email,Standard,Employer-Sponsored,-1,0.990,Follow-up scheduled,1,0,0,-1,15000,0,84107.08,0,1,NaN,57,NaN,NaN,NaN,20025.495238,2213.344211,4,0,Missing,Missing,-1.0
9936,15956,44,Male,Divorced,72928.63,Full-time,South,Yes,1.2,1,NaT,2024-01-02,Email,Standard,Third-Party,0,0.913,Requested more time,1,0,0,0,15000,0,72928.63,0,1,NaN,363,NaN,NaN,NaN,33149.377273,1657.468864,3,0,Missing,Missing,-1.0
9939,17914,50,Male,Married,60196.16,Part-time,Midwest,No,7.3,0,NaT,2024-08-15,SMS,Standard,Direct,-1,0.106,Declined - cost concern,1,0,0,-1,18000,0,60196.16,0,1,NaN,137,NaN,NaN,NaN,7252.549398,1203.923200,1,0,Missing,Missing,-1.0
9968,15438,23,Other,Married,71285.06,Full-time,Northeast,No,1.4,0,NaT,2024-01-19,Email,Standard,Direct,0,0.310,Spouse covered elsewhere,1,0,0,0,15000,0,71285.06,0,1,NaN,346,NaN,NaN,NaN,29702.108333,3099.350435,3,0,Missing,Missing,-1.0


FEATURE ENGINEERING---------------------------------------------------------------------------------

In [26]:
# ---------------- Date/channel feature engineering ----------------
def engineer_date_and_channel_features(df, reference_date=None):
    df = df.copy()
    if reference_date is None:
        candidates = [df["application_date"].max(), df["last_contact_date"].max()]
        candidates = [c for c in candidates if pd.notna(c)]
        reference_date = max(candidates) if candidates else pd.Timestamp.today()
    else:
        reference_date = pd.to_datetime(reference_date)
 
    df["days_since_application"] = (reference_date - df["application_date"]).dt.days
    df["days_since_last_contact"] = (reference_date - df["last_contact_date"]).dt.days
    df["contact_to_application_days"] = (df["application_date"] - df["last_contact_date"]).dt.days
    df["month_of_application"] = df["application_date"].dt.month
    df["day_of_week_application"] = df["application_date"].dt.dayofweek
    return df
 
employees_df = engineer_date_and_channel_features(employees_df)
print("\nSample engineered date features:")
print(employees_df[["application_date", "last_contact_date", "days_since_application",
                     "contact_to_application_days"]].head())
 


Sample engineered date features:
  application_date last_contact_date  days_since_application  \
0       2024-04-14        2024-03-30                   260.0   
1       2024-05-09        2024-04-18                   235.0   
2       2024-06-22        2024-06-09                   191.0   
3       2024-08-16        2024-07-29                   136.0   
4       2024-05-13        2024-05-06                   231.0   

   contact_to_application_days  
0                         15.0  
1                         21.0  
2                         13.0  
3                         18.0  
4                          7.0  


In [27]:
# ---------------- salary-derived features ----------------
employees_df["salary_per_tenure"] = employees_df["salary_clean"] / (employees_df["tenure_years"] + 1)
employees_df["salary_age_ratio"] = employees_df["salary_clean"] / employees_df["age"]
employees_df["salary_band"] = pd.qcut(employees_df["salary_clean"], q=5, labels=False, duplicates="drop")

In [28]:
# ---------------- region cleaning + join ----------------
employees_df["region"] = employees_df["region"].astype(str).str.strip().str.title()
region_df["region"] = region_df["region"].astype(str).str.strip().str.title()
 
mandate_mapping = {"low": 0, "med": 1, "medium": 1, "high": 2}
region_df["state_mandate_level_clean"] = (
    region_df["state_mandate_level"].astype(str).str.strip().str.lower().map(mandate_mapping)
)
print("\nRegion mandate mapping check:\n", region_df[["region", "state_mandate_level", "state_mandate_level_clean"]])
 
region_features = [
    "region", "n_employees_region", "avg_salary_region", "avg_premium_cost_usd",
    "benefits_broker_rating", "hr_outreach_capacity", "open_enrollment_window_days",
    "state_mandate_level_clean", "hist_enrollment_rate_region",
]
region_df_final = region_df[region_features].copy()
 
merged_df = employees_df.merge(region_df_final, on="region", how="left", validate="many_to_one")
assert merged_df["n_employees_region"].isna().sum() == 0, "unmatched region(s) after join"
print("\nShape after merge:", merged_df.shape)
 
recomputed = employees_df.groupby("region")["enrolled"].mean()
print(pd.concat([recomputed.rename("recomputed_current_rate"),
                  region_df_final.set_index("region")["hist_enrollment_rate_region"]], axis=1))
 


Region mandate mapping check:
       region state_mandate_level  state_mandate_level_clean
0    Midwest                High                          2
1  Northeast                 low                          0
2      South                 MED                          1
3       West                 Low                          0

Shape after merge: (9992, 43)
           recomputed_current_rate  hist_enrollment_rate_region
region                                                         
Midwest                   0.617363                        0.617
Northeast                 0.611422                        0.612
South                     0.627993                        0.628
West                      0.612878                        0.613


In [29]:
# ---------------- feature classification ----------------
feature_classification = {
    "usable": [
        "age", "employment_type", "tenure_years", "has_dependents",
        "salary_clean", "salary_band", "salary_per_tenure", "salary_age_ratio",
        "min_salary_allowed", "is_implausible_salary", "is_new_hire",
        "prior_year_enrolled_clean", "broker_channel",
        "last_contact_channel", "plan_tier_requested",
        "month_of_application", "day_of_week_application",
        "n_employees_region", "avg_salary_region", "avg_premium_cost_usd",
        "benefits_broker_rating", "hr_outreach_capacity", "open_enrollment_window_days",
        "state_mandate_level_clean", "hist_enrollment_rate_region",
    ],
    "analysis_only": [
        "has_missing_app_date", "is_contact_after_app",
        "legacy_propensity_score_missing", "has_outreach_note",
    ],
    "forbidden_leaky": [
        "application_date", "last_contact_date",
        "days_since_application", "days_since_last_contact", "contact_to_application_days",
        "legacy_propensity_score", "outreach_notes", "salary",
    ],
}
print("\nFeature classification:")
for bucket, feats in feature_classification.items():
    print(bucket, "->", feats)
 


Feature classification:
usable -> ['age', 'employment_type', 'tenure_years', 'has_dependents', 'salary_clean', 'salary_band', 'salary_per_tenure', 'salary_age_ratio', 'min_salary_allowed', 'is_implausible_salary', 'is_new_hire', 'prior_year_enrolled_clean', 'broker_channel', 'last_contact_channel', 'plan_tier_requested', 'month_of_application', 'day_of_week_application', 'n_employees_region', 'avg_salary_region', 'avg_premium_cost_usd', 'benefits_broker_rating', 'hr_outreach_capacity', 'open_enrollment_window_days', 'state_mandate_level_clean', 'hist_enrollment_rate_region']
analysis_only -> ['has_missing_app_date', 'is_contact_after_app', 'legacy_propensity_score_missing', 'has_outreach_note']
forbidden_leaky -> ['application_date', 'last_contact_date', 'days_since_application', 'days_since_last_contact', 'contact_to_application_days', 'legacy_propensity_score', 'outreach_notes', 'salary']


In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [35]:
X.isna().sum()[X.isna().sum() > 0]

month_of_application       719
day_of_week_application    719
dtype: int64

In [34]:
print("\nTotal rows with at least one NaN:")
print(X.isna().any(axis=1).sum())


Total rows with at least one NaN:
719


In [ ]:
# ---------------- fairness checkpoint ----------------
 
model_features = feature_classification["usable"]
X = pd.get_dummies(merged_df[model_features], drop_first=True)
y = merged_df["enrolled"]
 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, merged_df.index, test_size=0.3, random_state=42, stratify=y if y.nunique() > 1 else None
)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
preds = model.predict(X_test)
print("\nBaseline model report (demographics excluded from training):")
print(classification_report(y_test, preds, zero_division=0))
 
audit = merged_df.loc[idx_test, ["gender", "marital_status"]].copy()
audit["y_true"] = y_test.values
audit["y_pred"] = preds
for col in ["gender", "marital_status"]:
    print(f"\nSubgroup breakdown by {col}:")
    for grp, sub in audit.groupby(col):
        if len(sub) == 0:
            continue
        fnr = ((sub.y_true == 1) & (sub.y_pred == 0)).sum() / max((sub.y_true == 1).sum(), 1)
        fpr = ((sub.y_true == 0) & (sub.y_pred == 1)).sum() / max((sub.y_true == 0).sum(), 1)
        print(f"  {grp:10s} n={len(sub):3d}  FNR={fnr:.2f}  FPR={fpr:.2f}")
 

In [4]:
import pandas as pd
employees_processed = pd.read_csv('../data/processed/employees_processed.csv')

In [5]:
# Features to drop due to leakage, redundancy, or depending on future events
features_to_drop = [
    'legacy_propensity_score',
    'hist_enrollment_rate_region',
    'contact_to_application_days',
    'days_contact_to_app',
    'is_contact_after_app',
    'days_since_last_contact',
    'has_applied',
    'app_month',
    'app_day_of_week'
]

# Drop the features
employees_processed = employees_processed.drop(columns=features_to_drop, errors='ignore')

# Verify the changes
print(f"Dropped up to {len(features_to_drop)} features.")
print(f"Remaining columns: {employees_processed.shape[1]}")


Dropped up to 9 features.
Remaining columns: 31


In [10]:
import os
os.makedirs(PROC_DIR, exist_ok=True)
employees_processed.to_csv(f"{PROC_DIR}/employees_processed_no_leaky_features.csv", index=False)
print("\nExport complete.")


Export complete.


In [44]:
# ---------------- export ----------------
os.makedirs(PROC_DIR, exist_ok=True)
merged_df.to_csv(f"{PROC_DIR}/merged_df.csv", index=False)
employees_df.to_csv(f"{PROC_DIR}/employees_processed.csv", index=False)
region_df_final.to_csv(f"{PROC_DIR}/region_processed.csv", index=False)
print("\nExport complete.")
print(merged_df.shape)


Export complete.
(9992, 43)
